# US x1.0 — Canonical baseline model notebook

**Model ID:** `us_x1_0`  
**Role:** first named US selected-pool research baseline  
**Status:** `baseline_research`; `trade_ready=false`

This notebook is the human-readable companion to `configs/models/us_x1_0.yaml`. It explains the exact contract, complete development and frozen-challenge backtest evidence, reproduction commands, limitations and version-upgrade rules. It does not silently rerun a long model job when opened.

## 1. What US x1.0 is

US x1.0 ranks the governed US87 equity pool against QQQ using a 10-session XGBoost `rank:ndcg` model. It adopts the risk-controlled momentum candidate selected in PR #343 after six pre-registered development variants and one frozen 2026H1 challenge.

The version identifies an immutable model contract. Performance remains bound to the provider snapshot, dates and evidence artifact recorded below.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd
import yaml

def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "configs").is_dir():
            return candidate
    raise FileNotFoundError("Run this notebook inside the Alpha Engine repository.")

ROOT = find_repo_root()

MODEL_ID = "us_x1_0"
CONFIG_PATH = ROOT / "configs/models/us_x1_0.yaml"
REGISTRY_PATH = ROOT / "configs/models/model_registry_v1.yaml"

config = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))
registry = yaml.safe_load(REGISTRY_PATH.read_text(encoding="utf-8"))

assert config["model_id"] == MODEL_ID
assert config["display_name"] == "US x1.0"
assert config["trade_ready"] is False
assert registry["models"][MODEL_ID]["config"] == str(CONFIG_PATH.relative_to(ROOT))
config["display_name"], config["status"], config["provider_binding"]["canonical_evidence_provider_identity_sha256"]

## 2. Universe and data contract

- Universe: `us_selected_equities_v2`, 87 declared equities.
- Benchmark: QQQ, reference only; it never enters the stock rank.
- Membership: static curated, with explicit survivorship-bias disclosure.
- Listing policy: no fabricated pre-listing history; lifecycle exclusions must remain visible.
- Canonical evidence provider identity: `4921168fee3afdcfb222568bed370a0273a5d8795ac57e4477184c1728384a5c`.
- Provider cutoff: 2026-07-31.

In [ ]:
universe = config["universe"]
provider = config["provider_binding"]
pd.DataFrame(
    [
        {"field": "universe_id", "value": universe["universe_id"]},
        {"field": "declared_candidate_count", "value": universe["declared_candidate_count"]},
        {"field": "benchmark", "value": config["benchmark"]},
        {"field": "provider_identity", "value": provider["canonical_evidence_provider_identity_sha256"]},
        {"field": "provider_cutoff", "value": provider["cutoff"]},
        {"field": "snapshot_independent_claim", "value": provider["snapshot_independent_performance_claim"]},
    ]
)

## 3. Features, target and model parameters

The feature group is `risk_controlled_momentum`: three momentum horizons, two realized-volatility terms and two return-per-volatility terms. The training target is a daily cross-sectional percentile rank converted into seven relevance gains. Economic evaluation uses raw 10-session forward returns.

In [ ]:
pd.DataFrame(
    {"expression": config["features"]["expressions"]}
).rename_axis("feature_index").reset_index()

In [ ]:
model = config["model"]
label = config["label"]
strategy = config["strategy"]

pd.DataFrame(
    [
        {"parameter": "family", "value": model["family"]},
        {"parameter": "objective", "value": model["objective"]},
        {"parameter": "gain_bins", "value": label["gain_bins"]},
        {"parameter": "num_boost_round", "value": model["num_boost_round"]},
        {"parameter": "max_leaves", "value": model["max_leaves"]},
        {"parameter": "min_data_in_leaf", "value": model["min_data_in_leaf"]},
        {"parameter": "learning_rate", "value": model["learning_rate"]},
        {"parameter": "seed", "value": model["seed"]},
        {"parameter": "holding_sessions", "value": strategy["holding_sessions"]},
        {"parameter": "rebalance_sessions", "value": strategy["rebalance_sessions"]},
        {"parameter": "top_n", "value": strategy["top_n"]},
        {"parameter": "weighting", "value": strategy["weighting"]},
        {"parameter": "cost_bps", "value": strategy["cost_bps"]},
    ]
)

## 4. Backtest protocol and metric definitions

Development used the four complete windows 2024H1, 2024H2, 2025H1 and 2025H2. Candidate selection was completed before 2026H1 was evaluated once as the frozen challenge.

The authoritative compounded relative excess formula is:

`(1 + compounded_strategy_return) / (1 + compounded_benchmark_return) - 1`

Simple window excess shown below is `strategy_return - benchmark_return`; it is not substituted for the compounded formula.

In [ ]:
development = config["backtest_evidence"]["development"]
calculated_relative = (
    (1.0 + development["compounded_strategy_return"])
    / (1.0 + development["compounded_benchmark_return"])
    - 1.0
)
assert abs(calculated_relative - development["compounded_relative_excess_return"]) < 1e-10

pd.DataFrame(
    [
        {"metric": "compounded_strategy_return", "value": development["compounded_strategy_return"]},
        {"metric": "compounded_benchmark_return", "value": development["compounded_benchmark_return"]},
        {"metric": "compounded_relative_excess", "value": development["compounded_relative_excess_return"]},
        {"metric": "mean_icir", "value": development["mean_icir"]},
        {"metric": "mean_rank_ic", "value": development["mean_rank_ic"]},
        {"metric": "mean_top_bottom_spread", "value": development["mean_top_bottom_spread"]},
        {"metric": "positive_excess_windows", "value": development["positive_excess_windows"]},
        {"metric": "worst_drawdown", "value": development["worst_drawdown"]},
    ]
)

## 5. Complete development-window results

In [ ]:
windows = pd.DataFrame(development["windows"])
windows

Interpretation:

- Economics were positive in all four development windows.
- 2025H2 supplied a disproportionate share of the headline return.
- 2025H1 was the principal failure window, with negative ICIR and a -28.36% drawdown.
- Therefore total return alone is insufficient for promotion.

## 6. Frozen 2026H1 challenge

The challenge was consumed once in workflow run `30733686862`, artifact `8828827295`. It must not be reused for future candidate selection.

In [ ]:
challenge = config["backtest_evidence"]["frozen_challenge"]
pd.DataFrame([challenge]).T.rename(columns={0: "value"})

Compared with the prior US87 XGBoost baseline, US x1.0 improved 2026H1 excess return, ICIR, Rank IC, Top-Bottom spread and turnover. Maximum drawdown was 0.68 percentage points worse. The result supports research continuation, not trade readiness.

## 7. Evidence identity

In [ ]:
pd.DataFrame([config["evidence_identity"]]).T.rename(columns={0: "value"})

## 8. Reproduction

Contract validation is fast and safe. A complete backtest rebuilds the governed provider and reruns the frozen research spec. The full job is opt-in because it is computationally expensive.

In [ ]:
VALIDATE_COMMAND = [
    sys.executable,
    str(ROOT / "scripts/validate_model_x1_baselines.py"),
]
FULL_BACKTEST_COMMAND = [
    "uv", "run", "python", "scripts/run_us_feature_quality_validation.py",
    "--spec", "configs/research_paradigms/us_10d_xgb_optimization_frozen_v1.yaml",
    "--provider-uri", "artifacts/selected_pool_price_refresh/us/data/providers/us",
    "--output-dir", "artifacts/evidence/model_versions/us_x1_0",
]

print("Validate:", " ".join(VALIDATE_COMMAND))
print("Full backtest:", " ".join(FULL_BACKTEST_COMMAND))

RUN_CONTRACT_VALIDATION = False
RUN_FULL_BACKTEST = False

if RUN_CONTRACT_VALIDATION:
    subprocess.run(VALIDATE_COMMAND, cwd=ROOT, check=True)

if RUN_FULL_BACKTEST:
    subprocess.run(FULL_BACKTEST_COMMAND, cwd=ROOT, check=True)

## 9. Known limitations and next version

US x1.0 is immutable. Effective experiments may propose **US x1.1**, but cannot overwrite this contract.

Required evidence before a compatible minor-version promotion:

1. security, sector and rebalance-period contribution concentration;
2. seed and block-bootstrap rank/Top-15 stability;
3. 40/60 bps cost stress;
4. risk-control and Top-K robustness;
5. one new untouched challenge window.

The unresolved historical 81.43% figure is not an optimization target.

In [ ]:
pd.DataFrame({"known_limitation": config["known_limitations"]})